# 人工ナノシート領域分割デモ
# Synthetic Nanosheet Instance Segmentation Demo

---

## このノートブックで学ぶこと

このノートブックでは、人工的に生成したナノシート風画像を使って、画像領域分割の基本的な流れを体験します。

扱うタスクは、ナノシート画像中の個々のシートを分けて検出する **instance segmentation** です。

このデモでは、次の2種類の方法を比較します。

### ゼロショット分割ベースライン
学習を行わず、画像処理や基盤モデル風の候補マスク生成によって領域を抽出する方法です。
学習データを用意しなくても使える一方で、対象物に特化した調整が難しい場合があります。

### 学習済み分割モデル
人工ナノシート画像と正解マスクを用いて、対象物に合わせた分割モデルを学習した想定の方法です。
このデモでは、計算時間を短くするため、あらかじめ合成データだけから作成した予測結果を使います。

---

**重要：** このpublicデモには、実際の実験TEM画像、手動アノテーション、未公開の実験データは含まれていません。
すべての画像、マスク、予測結果は、教材用に生成した人工データに基づいています。

---

### 学ぶ流れ

1. 人工データを作る
2. 正解マスクを確認する
3. ゼロショット分割の結果を見る
4. 学習済みモデルの結果を見る
5. 評価指標を計算する
6. どの方法がよいかを比較する

実際の研究では、ここからさらに実験データへの転移（sim2real gap）、アノテーションの不完全性、重なり領域の扱い、評価指標の選び方などが重要になります。

## Step 1: GitHubから教材を取得する

In [ ]:
import os

repo_dir = "/content/nanosheet-segmentation-colab-demo"

if not os.path.exists(repo_dir):
    !git clone https://github.com/fanfanfuzzy/nanosheet-segmentation-colab-demo.git
else:
    print("Repository already exists. Skipping clone.")

%cd /content/nanosheet-segmentation-colab-demo

## Step 2: 必要なライブラリをインストールする

In [ ]:
!pip install -r requirements.txt -q

## Step 3: 人工ナノシート画像を生成する

ここでは、物理モデル（Beer-Lambert則に基づく透過率モデル）を使って、
ナノシート風の合成画像を生成します。

難易度設定（`configs/`）によって、ノイズ・コントラスト・重なり密度が変わります。
- `synthetic_easy.yaml`: 高コントラスト、低ノイズ
- `synthetic_mid.yaml`: 中程度
- `synthetic_hard.yaml`: 低コントラスト、高ノイズ、密な重なり

In [ ]:
!python src/generate_synthetic_nanosheets.py \
    --config configs/synthetic_mid.yaml \
    --num-images 10 \
    --output-dir outputs/synthetic_demo

## Step 4: 正解マスクを可視化する

生成した画像と、各ナノシートの正解マスク（instance mask）を重ねて表示します。
色が異なる領域が、それぞれ別のナノシートです。

In [ ]:
!python src/visualize_dataset.py \
    --input-dir outputs/synthetic_demo \
    --output-dir outputs/figures

In [ ]:
from IPython.display import Image, display
display(Image("outputs/figures/sample_dataset.png"))

## Step 5: ゼロショット分割の結果を見る

ここでは、学習を一切行わずに、画像処理ベースの手法でナノシート領域を抽出します。

使っている手法：
- 適応的閾値処理（adaptive thresholding）
- モルフォロジー処理（ノイズ除去）
- 距離変換 + 連結成分分析（インスタンス分離）

これは SAM（Segment Anything Model）のような「ゼロショット分割」のアイデアを
軽量に再現したものです。実際のSAMは使いません。

In [ ]:
!python src/sam_zero_shot_baseline.py \
    --input-dir outputs/synthetic_demo \
    --output-dir outputs/predictions_sam_baseline

## Step 6: 学習済みモデルの結果を見る

ここでは、合成データで学習したモデルの予測結果を読み込みます。

このデモでは計算時間を短縮するため、事前に生成した予測結果を使用します。
実際には、YOLO-seg や Mask R-CNN などのモデルを合成データで学習させます。

**注意：** このデモの学習済み結果はすべて合成データのみから作成されています。
実験データは一切使用していません。

In [ ]:
import shutil
import os

# Use pre-computed trained-model predictions from demo_assets
src_dir = "demo_assets/predictions_yolo_trained"
dst_dir = "outputs/predictions_yolo_trained"

if os.path.exists(src_dir) and os.listdir(src_dir):
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    print(f"Loaded pre-computed predictions from {src_dir}")
else:
    print("Pre-computed predictions not found.")
    print("Generating simulated trained-model predictions...")
    # Fallback: generate simulated better predictions
    !python src/sam_zero_shot_baseline.py \
        --input-dir outputs/synthetic_demo \
        --output-dir outputs/predictions_yolo_trained \
        --min-area 100 --max-area 80000

## Step 7: 評価指標を計算する

両方の手法の予測結果を、正解マスクと比較して評価します。

### 評価指標の意味

| 指標 | 意味 |
|------|------|
| `instance_recall` | 正解のナノシートのうち、モデルが見つけられた割合 |
| `instance_precision` | モデルが出した予測のうち、正解だった割合 |
| `instance_f1` | RecallとPrecisionの調和平均 |
| `mean_best_iou` | 各正解に対して最もよく重なる予測とのIoUの平均 |
| `semantic_iou` | 個体を区別せず、前景全体としてのIoU |

**注意：** 実データで正解アノテーションが一部しかない場合、precisionは低く見積もられることがあります。
モデルが本当に存在するナノシートを見つけても、GTに描かれていなければfalse positive扱いになるからです。

In [ ]:
# Evaluate zero-shot baseline
!python src/evaluate_predictions.py \
    --gt-dir outputs/synthetic_demo/masks \
    --pred-dir outputs/predictions_sam_baseline \
    --method-name sam_zero_shot \
    --output outputs/metrics_sam.csv

In [ ]:
# Evaluate trained model
!python src/evaluate_predictions.py \
    --gt-dir outputs/synthetic_demo/masks \
    --pred-dir outputs/predictions_yolo_trained \
    --method-name yolo_trained \
    --output outputs/metrics_yolo.csv

## Step 8: 結果を比較する

2つの手法の評価結果を並べて比較し、棒グラフで可視化します。

In [ ]:
!python src/compare_metrics.py \
    --inputs outputs/metrics_sam.csv outputs/metrics_yolo.csv \
    --output-csv outputs/comparison_metrics.csv \
    --output-fig outputs/comparison_barplot.png

In [ ]:
import pandas as pd
from IPython.display import Image, display

# Show comparison table
df = pd.read_csv("outputs/comparison_metrics.csv")
display(df)

# Show comparison bar chart
display(Image("outputs/comparison_barplot.png"))

## Step 9: 発展 — YOLOの短時間学習を試す（任意）

**このセルは実行しなくてもワークショップは完了します。**

GPUランタイムが利用可能な場合のみ、短いYOLO学習を体験できます。
実行には数分かかる場合があります。

```
ランタイム → ランタイムのタイプを変更 → GPU を選択
```

In [ ]:
# Optional: run a very short YOLO-style training demo.
# This cell may take time and may require a GPU runtime.
# Uncomment the lines below to try.

# !pip install ultralytics -q
# !yolo segment train \
#     model=yolo11n-seg.pt \
#     data=demo_assets/yolo_dataset.yaml \
#     epochs=3 \
#     imgsz=512 \
#     batch=4

## Step 10: まとめ

このデモでは、以下を学びました：

1. **人工ナノシート画像の生成** — 物理モデルに基づく合成データ作成
2. **ゼロショット分割** — 学習なしで候補マスクを生成する方法
3. **学習済みモデル** — タスク特化の学習により性能が向上すること
4. **評価指標** — instance recall, precision, IoU による定量評価
5. **手法の比較** — 棒グラフによる視覚的な性能比較

---

### 実際の研究では

実際の研究では、合成データで高性能だったモデルでも、実験画像に適用すると性能が低下する場合があります。
この差は **sim2real gap** と呼ばれます。

本publicデモでは、実験データそのものは公開せず、合成データを使って同じ評価の流れだけを体験しました。

---

### AI支援コーディングについて

このリポジトリの一部は、AIコーディングアシスタント（Devin）を使って作成されています。

重要なのは、AIにコードを書かせることよりも、
**AIが作った変更をGitHub上で確認し、安全に取り込むこと**です。

Pull Requestの「Files changed」タブで差分を確認する習慣をつけましょう。